In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="07-event-recommendation",
    notebook_name="01_event_recommendation_system_design.ipynb"
)

# Event Recommendation System Design

## The Big Idea (For a 12-Year-Old)

Imagine your phone could look at a million events happening around you -- concerts, basketball games, art shows, comedy nights -- and pick the ones YOU would love most. Not just popular ones, but ones that match YOUR taste, YOUR schedule, YOUR location, and even what YOUR friends are doing.

That is what we are building here. Think of it like a super-smart friend who:
- Knows you love hip-hop concerts but not country music
- Knows you usually go to events on weekends
- Knows your best friend just signed up for a basketball game
- Knows you do not like driving more than 30 minutes

And uses ALL of that to show you the perfect events.

---

## Staff-Level Technical Summary

We design an **Eventbrite-style event recommendation system** using:
- **Pointwise Learning-to-Rank** via binary classification
- **Five feature categories**: location, time, social, user, event
- **XGBoost baseline** graduating to **neural network** for continual learning
- **Two-pipeline architecture**: online learning + prediction (filtering -> ranking)
- **Offline evaluation**: mAP | **Online evaluation**: conversion rate, CTR, revenue lift

## 1. Problem Definition and Clarification

### The Interview Starts Here

In a real interview, you would ask clarifying questions before writing any code. Here is the full set of questions and answers:

In [ ]:
# Let's organize the clarifying requirements as structured data
# This is how you'd think about it systematically

requirements = {
    "business_objective": "Increase ticket sales (maximize event registrations)",
    "platform_scope": "Events only (no hotels, restaurants)",
    "event_nature": "Ephemeral -- one-time occurrence, expires after it happens",
    "event_attributes": ["description", "price", "location", "date_time", "category", "subcategory"],
    "training_data": "No hand-labeled data; construct from user interaction logs",
    "user_location": "Available (users consent to share)",
    "social_features": {
        "friendships": "Bidirectional (if A friends B, then B friends A)",
        "invitations": "Users can invite others to events",
        "registration": "Registration only (no RSVP)"
    },
    "pricing": "Both free and paid events",
    "scale": {
        "events_per_month": "~1 million",
        "daily_active_users": "~1 million",
    },
    "external_services": "Google Maps API / map services for distance and travel time"
}

print("=== Event Recommendation System Requirements ===")
print(f"\nBusiness Objective: {requirements['business_objective']}")
print(f"Scale: {requirements['scale']['events_per_month']} events/month, {requirements['scale']['daily_active_users']} DAU")
print(f"Event Nature: {requirements['event_nature']}")
print(f"\nKey Insight: Events are EPHEMERAL -- this is fundamentally different from")
print(f"movie/product recommendations where items persist forever.")

### Why Events Are Harder Than Movies or Products

| Challenge | Movie Recommendation | Event Recommendation |
|-----------|---------------------|---------------------|
| Item lifespan | Forever | Expires after event date |
| Cold start | Occasional (new movies) | **Constant** (new events daily) |
| Location | Irrelevant (streaming) | **Critical** (must be reachable) |
| Time sensitivity | Low | **Very high** (remaining time matters) |
| Social dynamics | Weak signal | **Strong signal** (friends attending) |
| Interaction history per item | Thousands of ratings | Very few (short-lived events) |

## 2. Framing as an ML Task

### ML Objective

**Business goal:** Increase ticket sales  
**ML objective:** Maximize event registrations  
**System I/O:** Input = user, Output = top-k events ranked by relevance

### Learning to Rank (LTR)

We reformulate this as a ranking problem using LTR. There are three approaches:

In [ ]:
# Understanding the three LTR approaches

ltr_approaches = {
    "Pointwise": {
        "how": "Predict relevance score for EACH item independently",
        "example_12yo": "Look at each event flyer and ask: 'Would Alice sign up? Yes/No?'",
        "algorithms": ["Logistic Regression", "Neural Network", "Any classifier"],
        "pros": "Simple to implement, can use any classifier",
        "cons": "Ignores relative ordering between items"
    },
    "Pairwise": {
        "how": "Given TWO items, predict which is more relevant",
        "example_12yo": "Hold up two event flyers: 'Which one would Alice prefer?'",
        "algorithms": ["RankNet", "LambdaRank", "LambdaMART"],
        "pros": "More accurate rankings",
        "cons": "Harder to implement, O(n^2) pairs"
    },
    "Listwise": {
        "how": "Predict optimal ordering of an ENTIRE list",
        "example_12yo": "Arrange ALL event flyers in the perfect order for Alice",
        "algorithms": ["SoftRank", "ListNet", "AdaRank"],
        "pros": "Directly optimizes the ranking metric",
        "cons": "Most complex to implement and train"
    }
}

print("=== Learning to Rank Approaches ===")
for approach, details in ltr_approaches.items():
    print(f"\n--- {approach} ---")
    print(f"  How: {details['how']}")
    print(f"  12yo: {details['example_12yo']}")
    print(f"  Algorithms: {', '.join(details['algorithms'])}")
    print(f"  Pros: {details['pros']}")
    print(f"  Cons: {details['cons']}")

print("\n==> We choose POINTWISE for simplicity: binary classification per (user, event) pair")

## 3. Metrics

### Offline Metrics

In [ ]:
import numpy as np

# === Offline Metrics for Ranking Systems ===

# Why NOT Precision@k / Recall@k?
# They don't consider the ORDERING -- just whether relevant items appear in top-k.
# For ranking, we need to reward putting relevant items HIGHER.

# Why NOT MRR (Mean Reciprocal Rank)?
# MRR focuses on the rank of the FIRST relevant item.
# In event recommendation, MULTIPLE events can be relevant.
# MRR would ignore all but the first one.

# Why mAP over nDCG?
# nDCG: works with graded relevance (e.g., 1-5 stars)
# mAP: works with BINARY relevance (registered or not)
# Since our labels are binary (registered=1, not=0), mAP is the best fit.

def average_precision(y_true, y_scores):
    """
    Compute Average Precision for a single query (user).
    
    Think of it this way (for a 12-year-old):
    - You ranked 10 events for Alice
    - Alice actually liked events at positions 1, 3, and 7
    - AP rewards you MORE for putting liked events near the TOP
    """
    # Sort by predicted scores (descending)
    sorted_indices = np.argsort(-y_scores)
    y_true_sorted = np.array(y_true)[sorted_indices]
    
    # Compute precision at each relevant position
    precisions = []
    relevant_count = 0
    
    for i, is_relevant in enumerate(y_true_sorted):
        if is_relevant == 1:
            relevant_count += 1
            precision_at_i = relevant_count / (i + 1)
            precisions.append(precision_at_i)
    
    if not precisions:
        return 0.0
    
    return np.mean(precisions)


def mean_average_precision(all_y_true, all_y_scores):
    """mAP = average of AP across all users."""
    aps = [average_precision(yt, ys) for yt, ys in zip(all_y_true, all_y_scores)]
    return np.mean(aps)


# Example: 3 users, each with 5 recommended events
# y_true: 1 = user registered, 0 = did not
# y_scores: model's predicted probability of registration

all_y_true = [
    [1, 0, 1, 0, 0],  # User A liked events 0 and 2
    [0, 1, 0, 0, 1],  # User B liked events 1 and 4
    [1, 1, 0, 0, 0],  # User C liked events 0 and 1
]
all_y_scores = [
    [0.9, 0.3, 0.8, 0.2, 0.1],  # Good: high scores for relevant items
    [0.1, 0.9, 0.3, 0.2, 0.8],  # Good: high scores for relevant items
    [0.7, 0.6, 0.9, 0.1, 0.05], # Decent: one irrelevant item ranked higher
]

mAP = mean_average_precision(all_y_true, all_y_scores)
print(f"Mean Average Precision (mAP): {mAP:.4f}")

# Show individual APs
for i, (yt, ys) in enumerate(zip(all_y_true, all_y_scores)):
    ap = average_precision(yt, ys)
    print(f"  User {chr(65+i)}: AP = {ap:.4f}")

In [ ]:
# === Online Metrics ===

def compute_online_metrics(impressions, clicks, registrations, bookmarks, revenue_before, revenue_after):
    """
    Compute all online metrics for the event recommendation system.
    
    12-year-old explanation:
    - CTR: "Out of every 100 events we showed you, how many did you click on?"
    - Conversion rate: "Out of every 100 events we showed you, how many did you actually sign up for?"
    - Bookmark rate: "Out of every 100 events, how many did you save for later?"
    - Revenue lift: "How much more money did we make after adding recommendations?"
    """
    metrics = {
        "CTR": clicks / impressions,
        "Conversion Rate": registrations / impressions,
        "Bookmark Rate": bookmarks / impressions,
        "Revenue Lift": (revenue_after - revenue_before) / revenue_before * 100
    }
    return metrics


# Example scenario
metrics = compute_online_metrics(
    impressions=100000,
    clicks=15000,
    registrations=3000,
    bookmarks=8000,
    revenue_before=500000,
    revenue_after=625000
)

print("=== Online Metrics ===")
for metric, value in metrics.items():
    if metric == "Revenue Lift":
        print(f"  {metric}: {value:.1f}%")
    else:
        print(f"  {metric}: {value:.4f} ({value*100:.1f}%)")

print("\nWhy not just CTR?")
print("  Some events are CLICKBAIT -- high clicks, low registrations.")
print("  Conversion rate is the metric that truly measures value.")
print("  Revenue lift directly measures business impact.")

## 4. High-Level Architecture

The system has two main pipelines:

```
                    EVENT RECOMMENDATION SYSTEM
    ========================================================
    
    PIPELINE 1: ONLINE LEARNING                    
    +-----------+     +-----------+     +----------+
    | New User  |---->| Feature   |---->| Model    |
    | Interactions|   | Pipeline  |     | Training |
    | (clicks,  |     | (compute  |     | (fine-   |
    |  registers,|    |  features)|     |  tune NN)|
    |  invites) |     +-----------+     +----+-----+
    +-----------+                            |
                                             v
                                       +----------+
                                       | Model    |
                                       | Eval &   |
                                       | Deploy   |
                                       +----+-----+
                                            |
    PIPELINE 2: PREDICTION                  v
    +------+    +----------+    +----------+    +--------+
    | User |    | Event    |    | Ranking  |    | Top-k  |
    | Query|--->| Filtering|--->| Service  |--->| Events |
    +------+    | (1M->100s)|   | (score & |    | to User|
                +----------+    |  sort)   |    +--------+
                                +----+-----+
                                     |
                           +---------+---------+
                           |                   |
                    +------+-----+    +--------+------+
                    | Feature    |    | Feature       |
                    | Store      |    | Computation   |
                    | (static)   |    | (dynamic,     |
                    |            |    |  real-time)   |
                    +------------+    +---------------+
```

## 5. Data Pipeline

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import random

# === Simulating the Raw Data ===

# Users table
users_data = {
    'user_id': [1, 2, 3, 4, 5],
    'username': ['alice', 'bob', 'charlie', 'diana', 'eve'],
    'age': [28, 35, 22, 45, 31],
    'gender': ['F', 'M', 'M', 'F', 'F'],
    'city': ['Miami', 'Miami', 'New York', 'Chicago', 'Miami'],
    'country': ['US', 'US', 'US', 'US', 'US'],
    'lat': [25.7617, 25.7617, 40.7128, 41.8781, 25.7617],
    'lon': [-80.1918, -80.1918, -74.0060, -87.6298, -80.1918]
}
users_df = pd.DataFrame(users_data)

# Events table
events_data = {
    'event_id': [101, 102, 103, 104, 105, 106],
    'host_user_id': [5, 3, 1, 2, 4, 3],
    'category': ['Music', 'Sports', 'Art', 'Music', 'Food', 'Tech'],
    'subcategory': ['Concert', 'Basketball', 'Theater', 'Jazz', 'Wine Tasting', 'AI Meetup'],
    'description': [
        'Dua Lipa Tour in Miami',
        'Warriors vs Bucks at MSG',
        'Comedy and Magic Show',
        'Jazz Night at Blue Note',
        'Wine and Cheese Festival',
        'Introduction to LLMs'
    ],
    'price': [200, 140, 0, 50, 75, 0],
    'city': ['Miami', 'New York', 'Miami', 'New York', 'Chicago', 'Miami'],
    'country': ['US', 'US', 'US', 'US', 'US', 'US'],
    'lat': [25.7814, 40.7505, 25.7903, 40.7306, 41.8827, 25.7750],
    'lon': [-80.1870, -73.9934, -80.1303, -74.0003, -87.6233, -80.1950],
    'event_datetime': [
        '2024-03-15 20:00', '2024-03-16 19:00', '2024-03-14 21:00',
        '2024-03-17 22:00', '2024-03-20 18:00', '2024-03-15 10:00'
    ],
    'walk_score': [92, 98, 75, 95, 60, 85]
}
events_df = pd.DataFrame(events_data)
events_df['event_datetime'] = pd.to_datetime(events_df['event_datetime'])

# Friendship table (bidirectional)
friendships_data = {
    'user_id_1': [1, 1, 2, 3, 4],
    'user_id_2': [2, 5, 3, 5, 5],
    'timestamp': [1658451341, 1659281720, 1659312942, 1660000000, 1660100000]
}
friendships_df = pd.DataFrame(friendships_data)

# Interactions table
interactions_data = {
    'user_id': [1, 1, 1, 2, 2, 3, 3, 4, 5, 5, 1, 2, 3, 4, 5, 1, 2, 3, 4, 5],
    'event_id': [101, 101, 102, 101, 103, 102, 104, 105, 101, 106, 103, 104, 105, 106, 102, 104, 105, 106, 101, 103],
    'interaction_type': [
        'impression', 'register', 'impression',
        'impression', 'impression',
        'impression', 'impression',
        'impression',
        'impression', 'impression',
        'impression', 'impression', 'impression', 'impression', 'impression',
        'register', 'register', 'register', 'impression', 'register'
    ]
}
interactions_df = pd.DataFrame(interactions_data)

print("=== Users ===")
print(users_df.to_string(index=False))
print("\n=== Events ===")
print(events_df[['event_id', 'category', 'subcategory', 'description', 'price', 'city']].to_string(index=False))
print("\n=== Friendships ===")
print(friendships_df.to_string(index=False))
print(f"\n=== Interactions ({len(interactions_df)} total) ===")
print(interactions_df.head(10).to_string(index=False))

In [ ]:
# === Constructing Training Dataset ===

# For each (user, event) impression pair:
#   label = 1 if user registered for the event
#   label = 0 if user saw the event but did NOT register

def construct_training_data(interactions_df):
    """
    Build training dataset from interaction logs.
    
    12-year-old version:
    For each event that was shown to a user, we ask:
    'Did they actually sign up?' If yes -> label 1. If no -> label 0.
    """
    # Get all impression pairs
    impressions = interactions_df[interactions_df['interaction_type'] == 'impression'][
        ['user_id', 'event_id']
    ].drop_duplicates()
    
    # Get all registration pairs
    registrations = interactions_df[interactions_df['interaction_type'] == 'register'][
        ['user_id', 'event_id']
    ].drop_duplicates()
    registrations['registered'] = 1
    
    # Merge: label = 1 if registered, 0 if only impression
    training_data = impressions.merge(registrations, on=['user_id', 'event_id'], how='left')
    training_data['label'] = training_data['registered'].fillna(0).astype(int)
    training_data = training_data.drop(columns=['registered'])
    
    return training_data

training_data = construct_training_data(interactions_df)

print("=== Training Dataset ===")
print(training_data.to_string(index=False))

# Show class imbalance
pos = training_data['label'].sum()
neg = len(training_data) - pos
print(f"\nPositive (registered): {pos} ({pos/len(training_data)*100:.1f}%)")
print(f"Negative (not registered): {neg} ({neg/len(training_data)*100:.1f}%)")
print(f"\nIn real systems, this ratio is much worse (~1:100 or worse).")
print(f"Solutions: focal loss, class-balanced loss, or undersample majority class.")

## 6. Feature Engineering Overview

This is the **most important part** of the design. Because events are ephemeral and cold-start is constant, we cannot rely on collaborative filtering alone -- we need rich, hand-crafted features.

### The Five Feature Categories

```
                        FEATURE CATEGORIES
    =====================================================
    
    +------------------+  +------------------+  +------------------+
    | 1. LOCATION      |  | 2. TIME          |  | 3. SOCIAL        |
    | - Walk score     |  | - Remaining time |  | - Friends going  |
    | - Same city?     |  | - Day-of-week    |  | - Invitations    |
    | - Distance       |  | - Hour-of-day    |  | - Host is friend?|
    | - Transit score  |  | - Travel time    |  | - # registered   |
    +------------------+  +------------------+  +------------------+
    
    +------------------+  +------------------+
    | 4. USER          |  | 5. EVENT         |
    | - Age bucket     |  | - Price bucket   |
    | - Gender         |  | - Price sim.     |
    |                  |  | - Description    |
    |                  |  |   similarity     |
    +------------------+  +------------------+
    
    Plus "SIMILARITY" features for almost everything:
    Compare the current event's feature to the user's
    historical average from past registered events.
```

In [ ]:
from math import radians, cos, sin, asin, sqrt

# === Feature Engineering: The Complete Pipeline ===

def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance between two points in miles.
    
    12-year-old version: 'How far is this event from where I am, 
    as the crow flies?'
    """
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    miles = 3956 * c  # Earth radius in miles
    return miles


def bucketize_distance(distance_miles):
    """Bucketize distance into categories."""
    if distance_miles < 1:
        return 0  # Less than a mile
    elif distance_miles < 5:
        return 1  # 1-5 miles
    elif distance_miles < 20:
        return 2  # 5-20 miles
    elif distance_miles < 50:
        return 3  # 20-50 miles
    elif distance_miles < 100:
        return 4  # 50-100 miles
    else:
        return 5  # 100+ miles


def bucketize_walk_score(score):
    """Bucketize walk score into categories."""
    if score >= 90:
        return 1  # No car needed
    elif score >= 70:
        return 2  # Very walkable
    elif score >= 50:
        return 3  # Somewhat walkable
    elif score >= 25:
        return 4  # Car-dependent
    else:
        return 5  # Requires a car


def bucketize_price(price):
    """Bucketize event price."""
    if price == 0:
        return 0  # Free
    elif price < 100:
        return 1  # $1-99
    elif price < 500:
        return 2  # $100-499
    elif price < 2000:
        return 3  # $500-1999
    else:
        return 4  # $2000+


def bucketize_remaining_time(hours):
    """Bucketize remaining time until event."""
    if hours < 1:
        return 0
    elif hours < 2:
        return 1
    elif hours < 4:
        return 2
    elif hours < 6:
        return 3
    elif hours < 12:
        return 4
    elif hours < 24:
        return 5
    elif hours < 72:
        return 6  # 1-3 days
    elif hours < 168:
        return 7  # 3-7 days
    else:
        return 8  # 7+ days


# Demonstrate the feature pipeline on one (user, event) pair
user = users_df[users_df['user_id'] == 1].iloc[0]
event = events_df[events_df['event_id'] == 101].iloc[0]
current_time = datetime(2024, 3, 13, 14, 0)  # Simulate "now"

# Location features
dist = haversine(user['lat'], user['lon'], event['lat'], event['lon'])
dist_bucket = bucketize_distance(dist)
same_city = int(user['city'] == event['city'])
same_country = int(user['country'] == event['country'])
walk_score_bucket = bucketize_walk_score(event['walk_score'])

# Time features
remaining_hours = (event['event_datetime'] - current_time).total_seconds() / 3600
remaining_bucket = bucketize_remaining_time(remaining_hours)
event_day = event['event_datetime'].weekday()  # 0=Monday, 6=Sunday
event_hour = event['event_datetime'].hour

# Event features
price_bucket = bucketize_price(event['price'])

print(f"=== Features for User {user['username']} x Event '{event['description']}' ===")
print(f"\n--- Location Features ---")
print(f"  Distance: {dist:.2f} miles (bucket: {dist_bucket})")
print(f"  Same city: {same_city}")
print(f"  Same country: {same_country}")
print(f"  Walk score: {event['walk_score']} (bucket: {walk_score_bucket})")

print(f"\n--- Time Features ---")
print(f"  Remaining time: {remaining_hours:.1f} hours (bucket: {remaining_bucket})")
print(f"  Day of week: {event_day} ({'Mon Tue Wed Thu Fri Sat Sun'.split()[event_day]})")
print(f"  Hour: {event_hour}")

print(f"\n--- Event Features ---")
print(f"  Price: ${event['price']} (bucket: {price_bucket})")

In [ ]:
# === Complete Feature Vector Construction ===

def get_user_friends(user_id, friendships_df):
    """Get all friends for a user (bidirectional friendships)."""
    friends_1 = friendships_df[friendships_df['user_id_1'] == user_id]['user_id_2'].tolist()
    friends_2 = friendships_df[friendships_df['user_id_2'] == user_id]['user_id_1'].tolist()
    return set(friends_1 + friends_2)


def get_registered_users(event_id, interactions_df):
    """Get set of users who registered for an event."""
    regs = interactions_df[
        (interactions_df['event_id'] == event_id) & 
        (interactions_df['interaction_type'] == 'register')
    ]
    return set(regs['user_id'].tolist())


def compute_social_features(user_id, event_id, events_df, friendships_df, interactions_df):
    """
    Compute social features for a (user, event) pair.
    
    12-year-old version: 'Are my friends going? Did someone invite me?
    Is the event host my friend?'
    """
    friends = get_user_friends(user_id, friendships_df)
    registered = get_registered_users(event_id, interactions_df)
    event = events_df[events_df['event_id'] == event_id].iloc[0]
    
    # Total registered
    num_registered = len(registered)
    
    # Impressions for this event
    num_impressions = len(interactions_df[
        (interactions_df['event_id'] == event_id) & 
        (interactions_df['interaction_type'] == 'impression')
    ])
    
    registration_ratio = num_registered / max(num_impressions, 1)
    
    # Friends registered
    friends_registered = friends & registered
    num_friends_registered = len(friends_registered)
    friend_reg_ratio = num_friends_registered / max(len(friends), 1)
    
    # Is host a friend?
    host_is_friend = int(event['host_user_id'] in friends)
    
    return {
        'num_registered': num_registered,
        'registration_ratio': round(registration_ratio, 4),
        'num_friends_registered': num_friends_registered,
        'friend_registration_ratio': round(friend_reg_ratio, 4),
        'host_is_friend': host_is_friend
    }


# Demo: social features for user 1, event 101
social_feats = compute_social_features(1, 101, events_df, friendships_df, interactions_df)

print(f"=== Social Features for User 1 (Alice) x Event 101 (Dua Lipa) ===")
print(f"  Alice's friends: {get_user_friends(1, friendships_df)}")
print(f"  Users registered for event 101: {get_registered_users(101, interactions_df)}")
for feat, val in social_feats.items():
    print(f"  {feat}: {val}")

## 7. Model Architecture

### Model Selection: The Journey from Simple to Complex

In [ ]:
# === Model Comparison ===

models = {
    "Logistic Regression": {
        "pros": ["Fast inference", "Fast training", "Interpretable", "Simple"],
        "cons": ["Cannot learn non-linear relationships", "Multicollinearity issues"],
        "verdict": "Too simple for complex feature interactions"
    },
    "Decision Tree": {
        "pros": ["Fast", "No normalization needed", "Interpretable"],
        "cons": ["Overfits easily", "Axis-parallel boundaries", "Sensitive to data changes"],
        "verdict": "Too unstable; rarely used alone"
    },
    "Random Forest (Bagging)": {
        "pros": ["Reduces variance/overfitting", "Parallel training"],
        "cons": ["Does not reduce bias", "No continual learning"],
        "verdict": "Better, but does not help with underfitting"
    },
    "GBDT / XGBoost (Boosting)": {
        "pros": ["Reduces bias AND variance", "Great with structured data", "Fast to implement"],
        "cons": ["Many hyperparameters", "No continual learning", "Bad with unstructured data"],
        "verdict": "GOOD BASELINE -- start here"
    },
    "Neural Network": {
        "pros": ["Continual learning", "Learns non-linear patterns", "Expressive"],
        "cons": ["Expensive to train", "Needs data preparation", "Black box"],
        "verdict": "PRODUCTION MODEL -- critical for adapting to new events"
    }
}

print("=== Model Selection Guide ===")
for model, details in models.items():
    print(f"\n{'='*50}")
    print(f"  {model}")
    print(f"{'='*50}")
    print(f"  Pros: {', '.join(details['pros'])}")
    print(f"  Cons: {', '.join(details['cons'])}")
    print(f"  >>> {details['verdict']}")

In [ ]:
import numpy as np

# === XGBoost Baseline Implementation ===

# Simulate feature matrix for demonstration
np.random.seed(42)
n_samples = 1000

# Create synthetic features matching our feature engineering
feature_names = [
    # Location features
    'distance_bucket', 'same_city', 'same_country', 'walk_score_bucket', 'walk_score_sim',
    # Time features  
    'remaining_time_bucket', 'remaining_time_sim', 'travel_time_bucket', 'day_match', 'hour_match',
    # Social features
    'num_registered', 'registration_ratio', 'num_friends_registered', 'friend_reg_ratio', 'host_is_friend',
    # User features
    'age_bucket', 'gender_female',
    # Event features
    'price_bucket', 'price_similarity', 'description_similarity'
]

X = np.random.randn(n_samples, len(feature_names))

# Generate labels with class imbalance (only ~10% positive)
# Registration probability is influenced by features
logits = (
    -2.0  # base (creates imbalance)
    + 0.5 * X[:, feature_names.index('same_city')]
    + 0.3 * X[:, feature_names.index('num_friends_registered')]
    + 0.2 * X[:, feature_names.index('description_similarity')]
    - 0.3 * X[:, feature_names.index('distance_bucket')]
    + 0.4 * X[:, feature_names.index('host_is_friend')]
)
probs = 1 / (1 + np.exp(-logits))
y = (np.random.rand(n_samples) < probs).astype(int)

print(f"Dataset: {n_samples} samples, {len(feature_names)} features")
print(f"Class distribution: {np.mean(y)*100:.1f}% positive, {(1-np.mean(y))*100:.1f}% negative")
print(f"\nFeatures ({len(feature_names)}):")
for i, name in enumerate(feature_names):
    print(f"  {i+1:2d}. {name}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features (needed for LR and NN, not for trees)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- Model 1: Logistic Regression (Baseline) ---
lr_model = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model.fit(X_train_scaled, y_train)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_auc = roc_auc_score(y_test, lr_probs)

# --- Model 2: GBDT (Gradient Boosted Decision Trees) ---
gbdt_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gbdt_model.fit(X_train, y_train)  # No scaling needed for trees!
gbdt_probs = gbdt_model.predict_proba(X_test)[:, 1]
gbdt_auc = roc_auc_score(y_test, gbdt_probs)

print("=== Model Comparison ===")
print(f"\nLogistic Regression AUC: {lr_auc:.4f}")
print(f"GBDT AUC:                {gbdt_auc:.4f}")

# Show feature importance from GBDT
print(f"\n=== Top Features (GBDT Feature Importance) ===")
importances = gbdt_model.feature_importances_
sorted_idx = np.argsort(importances)[::-1]
for i, idx in enumerate(sorted_idx[:10]):
    print(f"  {i+1}. {feature_names[idx]}: {importances[idx]:.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# === Neural Network Model (Production) ===

class EventRecommendationNN(nn.Module):
    """
    Neural network for event recommendation.
    
    12-year-old version:
    This is like a brain with layers. The first layer sees the raw features,
    the middle layers figure out complex patterns (like 'users who live
    in Miami AND have friends going to music events tend to register'),
    and the final layer gives a yes/no prediction.
    
    Staff-level detail:
    - Input: concatenated feature vector (location + time + social + user + event)
    - Architecture: MLP with batch normalization and dropout
    - Output: sigmoid probability of registration
    - Key advantage: supports continual learning via fine-tuning
    """
    def __init__(self, input_dim, hidden_dims=[128, 64, 32]):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3)
            ])
            prev_dim = hidden_dim
        
        # Final binary classification layer
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze(-1)


# Training setup
input_dim = len(feature_names)
model = EventRecommendationNN(input_dim)

# Use Binary Cross Entropy loss (standard for binary classification)
# With class weights to handle imbalance
pos_weight = torch.tensor([(1 - np.mean(y_train)) / np.mean(y_train)])
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)  # Will remove sigmoid in forward for this

# Actually, since we already have sigmoid, use BCELoss
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Prepare data
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Training loop
print("=== Training Neural Network ===")
print(f"Architecture: {input_dim} -> 128 -> 64 -> 32 -> 1")
print(f"Loss: Binary Cross Entropy")
print(f"Optimizer: Adam (lr=0.001)")
print()

model.train()
for epoch in range(20):
    total_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if (epoch + 1) % 5 == 0:
        model.eval()
        with torch.no_grad():
            test_probs = model(X_test_tensor).numpy()
            test_auc = roc_auc_score(y_test, test_probs)
        print(f"Epoch {epoch+1:3d} | Loss: {total_loss/len(train_loader):.4f} | Test AUC: {test_auc:.4f}")
        model.train()

# Final evaluation
model.eval()
with torch.no_grad():
    nn_probs = model(X_test_tensor).numpy()
    nn_auc = roc_auc_score(y_test, nn_probs)

print(f"\n=== Final Comparison ===")
print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"GBDT AUC:                {gbdt_auc:.4f}")
print(f"Neural Network AUC:      {nn_auc:.4f}")
print(f"\nNote: With real data and more training, NN typically outperforms GBDT")
print(f"Key advantage of NN: supports CONTINUAL LEARNING for adapting to new events")

In [ ]:
# === Continual Learning Demonstration ===

def simulate_continual_learning(model, optimizer, criterion, new_data_batches, X_test_tensor, y_test):
    """
    Demonstrate how the NN can be fine-tuned on new data.
    
    12-year-old version:
    Every day, new events appear and new users sign up.
    The NN can keep learning from this new data without
    starting from scratch -- like how you get better at a
    video game the more you play, without resetting your progress.
    
    Staff-level detail:
    Unlike GBDT which requires full retraining (expensive at scale),
    NNs support fine-tuning on new data batches. This is critical
    for event recommendation because:
    1. New events appear constantly (cold start)
    2. User preferences change over time (concept drift)
    3. Retraining GBDT from scratch is too costly
    """
    print("=== Continual Learning Simulation ===")
    print("Simulating 5 days of new data arriving...\n")
    
    for day, (new_X, new_y) in enumerate(new_data_batches):
        # Fine-tune on new data
        model.train()
        new_X_tensor = torch.FloatTensor(new_X)
        new_y_tensor = torch.FloatTensor(new_y)
        
        for _ in range(3):  # Few epochs of fine-tuning
            optimizer.zero_grad()
            pred = model(new_X_tensor)
            loss = criterion(pred, new_y_tensor)
            loss.backward()
            optimizer.step()
        
        # Evaluate
        model.eval()
        with torch.no_grad():
            test_probs = model(X_test_tensor).numpy()
            auc = roc_auc_score(y_test, test_probs)
        
        print(f"  Day {day+1}: Fine-tuned on {len(new_X)} new samples | Test AUC: {auc:.4f}")
    
    print("\nKey insight: The model improves as it sees more data,")
    print("WITHOUT retraining from scratch. GBDT cannot do this.")


# Generate simulated new data batches (5 days)
new_data_batches = []
for _ in range(5):
    new_X = scaler.transform(np.random.randn(50, len(feature_names)))
    new_logits = -2.0 + 0.5 * new_X[:, 1] + 0.3 * new_X[:, 12]
    new_probs = 1 / (1 + np.exp(-new_logits))
    new_y = (np.random.rand(50) < new_probs).astype(float)
    new_data_batches.append((new_X, new_y))

simulate_continual_learning(model, optimizer, criterion, new_data_batches, X_test_tensor, y_test)

## 8. System Architecture Diagram

### Prediction Pipeline: Step by Step

```
User Opens App
      |
      v
+-----+------+
| EVENT       |  Input: 1 million total events
| FILTERING   |  Rules: location, category, user filters
|             |  Output: ~100-500 candidate events
+-----+------+
      |  (e.g., "concerts in Miami in next 7 days")
      v
+-----+------+
| FEATURE     |  For each (user, event) pair:
| COMPUTATION |  - Static features from Feature Store
|             |  - Dynamic features computed real-time
+-----+------+
      |
      v
+-----+------+
| RANKING     |  Model predicts P(register) for each pair
| MODEL       |  Sort by probability descending
|             |  Return top-k events
+-----+------+
      |
      v
 Top-k Recommended Events --> User's Screen
```

### Online Learning Pipeline

```
New Interactions (clicks, registrations, new events, new users)
      |
      v
+-----+------+
| DATA        |  Compute features for new interactions
| PIPELINE    |  Add to training dataset
+-----+------+
      |
      v
+-----+------+
| CONTINUAL   |  Fine-tune NN on new data
| TRAINING    |  (No need to retrain from scratch!)
+-----+------+
      |
      v
+-----+------+
| EVAL &      |  Evaluate on held-out set
| DEPLOY      |  If better -> deploy to production
+-------------+
```

In [ ]:
# === End-to-End Prediction Pipeline Simulation ===

class EventRecommendationPipeline:
    """
    Complete prediction pipeline for event recommendations.
    
    12-year-old version:
    1. Start with ALL events (like a giant pile of flyers)
    2. Throw away ones that are too far, wrong category, etc.
    3. For the remaining ones, compute a 'would you like this?' score
    4. Show the top ones to the user
    """
    
    def __init__(self, model, scaler, events_df, users_df, friendships_df, interactions_df):
        self.model = model
        self.scaler = scaler
        self.events_df = events_df
        self.users_df = users_df
        self.friendships_df = friendships_df
        self.interactions_df = interactions_df
    
    def filter_events(self, user_id, max_distance_miles=50, category_filter=None):
        """
        Step 1: Filter events by simple rules.
        Reduces 1M events to hundreds of candidates.
        """
        user = self.users_df[self.users_df['user_id'] == user_id].iloc[0]
        candidates = []
        
        for _, event in self.events_df.iterrows():
            # Filter by distance
            dist = haversine(user['lat'], user['lon'], event['lat'], event['lon'])
            if dist > max_distance_miles:
                continue
            
            # Filter by category
            if category_filter and event['category'] != category_filter:
                continue
            
            # Filter expired events
            if event['event_datetime'] < datetime.now():
                continue  # Skip -- event already happened!
            
            candidates.append(event)
        
        return candidates
    
    def compute_features_for_pair(self, user_id, event):
        """
        Step 2: Compute feature vector for a (user, event) pair.
        Combines static features (from store) and dynamic features (real-time).
        """
        user = self.users_df[self.users_df['user_id'] == user_id].iloc[0]
        
        # For demo, generate synthetic features
        # In production, these come from the feature engineering pipeline
        np.random.seed(hash((user_id, event['event_id'])) % 2**31)
        features = np.random.randn(len(feature_names))
        
        # Override some features with real computed values
        dist = haversine(user['lat'], user['lon'], event['lat'], event['lon'])
        features[feature_names.index('distance_bucket')] = bucketize_distance(dist)
        features[feature_names.index('same_city')] = float(user['city'] == event['city'])
        features[feature_names.index('same_country')] = float(user['country'] == event['country'])
        features[feature_names.index('price_bucket')] = bucketize_price(event['price'])
        
        return features
    
    def rank_events(self, user_id, candidates, top_k=5):
        """
        Step 3: Score and rank candidate events.
        """
        if not candidates:
            return []
        
        # Compute features for all candidates
        feature_matrix = np.array([
            self.compute_features_for_pair(user_id, event)
            for event in candidates
        ])
        
        # Scale and predict
        feature_matrix_scaled = self.scaler.transform(feature_matrix)
        self.model.eval()
        with torch.no_grad():
            scores = self.model(torch.FloatTensor(feature_matrix_scaled)).numpy()
        
        # Sort by score
        ranked_indices = np.argsort(-scores)
        
        results = []
        for idx in ranked_indices[:top_k]:
            event = candidates[idx]
            results.append({
                'event_id': event['event_id'],
                'description': event['description'],
                'score': float(scores[idx]),
                'category': event['category'],
                'price': event['price']
            })
        
        return results
    
    def recommend(self, user_id, top_k=5, max_distance=50, category=None):
        """
        Full pipeline: filter -> feature compute -> rank -> return top-k.
        """
        # Step 1: Filter
        candidates = self.filter_events(user_id, max_distance, category)
        
        # Step 2-3: Feature compute + Rank
        ranked = self.rank_events(user_id, candidates, top_k)
        
        return ranked


# Demo the pipeline
pipeline = EventRecommendationPipeline(
    model=model,
    scaler=scaler,
    events_df=events_df,
    users_df=users_df,
    friendships_df=friendships_df,
    interactions_df=interactions_df
)

# Note: Since our sample events are in 2024, they are in the past.
# Let's adjust for demo purposes by using filter_events with a large distance and no date filter.
candidates = list(events_df.iterrows())
candidates = [row for _, row in events_df.iterrows()]

# Manually rank
results = pipeline.rank_events(user_id=1, candidates=candidates, top_k=5)

print("=== Recommendations for User 1 (Alice, Miami) ===")
for i, rec in enumerate(results):
    print(f"  #{i+1}: {rec['description']}")
    print(f"       Category: {rec['category']} | Price: ${rec['price']} | Score: {rec['score']:.4f}")

## Key Takeaways

1. **Events are ephemeral** -- this makes the problem fundamentally different from movie/product recs
2. **Feature engineering is king** -- 5 categories (location, time, social, user, event) with similarity features
3. **Pointwise LTR** -- binary classification for each (user, event) pair
4. **Start with XGBoost, graduate to NN** -- NN enables continual learning which is critical
5. **Two-pipeline architecture** -- online learning (continuous training) + prediction (filter -> rank)
6. **mAP offline, conversion rate online** -- binary relevance calls for mAP; conversion rate measures real business value
7. **Class imbalance** -- address with focal loss, class-balanced loss, or undersampling

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)